# Unit 4: Machine Learning — Quantum Kernel Classifier

**Companion notebook for *What Quantum Computers Are Actually For***

We build a quantum kernel classifier: encode classical data into quantum states on a cloud
[Quokka](https://www.quokkacomputing.com/), compute the kernel matrix from measurement
overlaps, and train a classical SVM.

**What you'll see:**
1. Generate a synthetic 2D dataset (two interleaved half-moons)
2. Define a quantum feature map (parameterised rotations + entanglement)
3. Compute the quantum kernel matrix on Quokka
4. Train an SVM with the quantum kernel
5. Compare with a classical RBF kernel

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. Generate data

In [ ]:
# Two interleaved half-moons
X, y = make_moons(n_samples=40, noise=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='#009688', label='Class 0', edgecolors='k', s=60)
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='#FF5722', label='Class 1', edgecolors='k', s=60)
plt.title('Training data (two half-moons)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Training: {len(X_train)} points, Test: {len(X_test)} points")

## 2. Quantum feature map

Encode a 2D data point $(x_1, x_2)$ into a 2-qubit state using:
1. $R_Y(x_1)$ and $R_Y(x_2)$ — encode the features
2. CNOT — entangle
3. $R_Z(x_1 \cdot x_2)$ — encode feature interactions

The kernel $K(x, x') = |\langle 0 | U^\dagger(x') U(x) | 0 \rangle|^2$ is the probability of
measuring $|00\rangle$ after applying $U(x)$ then $U^\dagger(x')$.

In [ ]:
def feature_map_circuit(x: np.ndarray) -> str:
    """Build the feature map U(x) as QASM gates."""
    x1, x2 = float(x[0]), float(x[1])
    return f"""ry({x1:.6f}) q[0];
ry({x2:.6f}) q[1];
cx q[0], q[1];
rz({x1 * x2:.6f}) q[1];
cx q[0], q[1];"""


def feature_map_adjoint(x: np.ndarray) -> str:
    """Build U†(x) — the adjoint (reverse order, negate angles)."""
    x1, x2 = float(x[0]), float(x[1])
    return f"""cx q[0], q[1];
rz({-x1 * x2:.6f}) q[1];
cx q[0], q[1];
ry({-x2:.6f}) q[1];
ry({-x1:.6f}) q[0];"""


def kernel_circuit(x: np.ndarray, xp: np.ndarray) -> str:
    """Build the kernel estimation circuit: U(x) then U†(x')."""
    return f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Feature map U(x)
{feature_map_circuit(x)}

// Adjoint feature map U†(x')
{feature_map_adjoint(xp)}

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

# Show an example
print(kernel_circuit(X_train[0], X_train[1]))

## 3. Compute the quantum kernel matrix

In [ ]:
def quantum_kernel_value(x, xp, shots=512):
    """Compute K(x, x') = Pr(|00⟩) from the kernel circuit."""
    circuit = kernel_circuit(x, xp)
    counts = run_qasm(circuit, shots=shots)
    total = sum(counts.values())
    p_00 = counts.get('00', 0) / total
    return p_00


def quantum_kernel_matrix(X_a, X_b, shots=512):
    """Compute the kernel matrix K[i,j] = K(X_a[i], X_b[j])."""
    n_a, n_b = len(X_a), len(X_b)
    K = np.zeros((n_a, n_b))
    total = n_a * n_b
    for i in range(n_a):
        for j in range(n_b):
            K[i, j] = quantum_kernel_value(X_a[i], X_b[j], shots=shots)
        print(f"  Row {i+1}/{n_a} done ({(i+1)*n_b}/{total} circuits)")
    return K


print(f"Computing {len(X_train)}×{len(X_train)} training kernel matrix...")
K_train = quantum_kernel_matrix(X_train, X_train, shots=512)

print(f"\nComputing {len(X_test)}×{len(X_train)} test kernel matrix...")
K_test = quantum_kernel_matrix(X_test, X_train, shots=512)

print("Done!")

In [ ]:
# Visualise the kernel matrix
plt.figure(figsize=(6, 5))
plt.imshow(K_train, cmap='viridis')
plt.colorbar(label='K(x, x\')')
plt.title('Quantum Kernel Matrix (training set)')
plt.xlabel('Sample index')
plt.ylabel('Sample index')
plt.tight_layout()
plt.show()

## 4. Train SVM with quantum kernel vs. classical RBF

In [ ]:
# Quantum kernel SVM
svm_quantum = SVC(kernel='precomputed')
svm_quantum.fit(K_train, y_train)
y_pred_quantum = svm_quantum.predict(K_test)
acc_quantum = accuracy_score(y_test, y_pred_quantum)

# Classical RBF kernel SVM
svm_rbf = SVC(kernel='rbf', gamma='auto')
svm_rbf.fit(X_train, y_train)
y_pred_rbf = svm_rbf.predict(X_test)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

print(f"Quantum kernel SVM accuracy: {acc_quantum:.1%}")
print(f"Classical RBF SVM accuracy:  {acc_rbf:.1%}")

## What's next?

- Try different feature maps (more layers, different entanglement patterns)
- Increase the dataset size and see how kernel quality scales
- Compare with a linear kernel to see when non-linearity matters
- [Unit 5 — Finance](../manuscript/05-finance.md): quantum sampling for Monte Carlo